# Colab Remote Kernel Setup

Run this notebook in Google Colab with a GPU runtime enabled. It starts a Jupyter server inside Colab and exposes it so VS Code can attach to the remote kernel. The setup tries ngrok first and automatically falls back to a Cloudflare quick tunnel if ngrok is not configured.

In [1]:
!pip install -q pyngrok jupyterlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 125.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 101.7 MB/s eta 0:00:00


## Optional: set your ngrok authtoken

This step is optional. If you do not configure ngrok, the startup cell will fall back automatically to a Cloudflare quick tunnel. If you want to use ngrok explicitly, uncomment the next line and replace the token. You can get it from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# !ngrok config add-authtoken "YOUR_NGROK_TOKEN"

In [3]:
import re
import socket
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path

from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError

try:
    import google.colab  # type: ignore
except ImportError as exc:
    raise RuntimeError(
        "This notebook cell must be run inside a Google Colab runtime, not in the local VS Code kernel. "
        "Open notebooks/00_colab_remote_kernel_setup.ipynb in the Colab browser, enable a GPU runtime there, and run the cells in Colab first. "
        "Only after Cell 5 prints a remote Jupyter URL should you attach VS Code to that URL."
    ) from exc

PORT = 8888
TOKEN = "gpu"
RUNTIME_DIR = Path(tempfile.gettempdir()) / "vscode-colab"
LOG_PATH = RUNTIME_DIR / "jupyter-vscode.log"
CLOUDFLARED_PATH = RUNTIME_DIR / "cloudflared"
CLOUDFLARED_LOG_PATH = RUNTIME_DIR / "cloudflared.log"


def wait_for_port(host: str, port: int, timeout: float = 30.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket() as sock:
            sock.settimeout(1.0)
            try:
                sock.connect((host, port))
                return True
            except OSError:
                if server.poll() is not None:
                    return False
                time.sleep(0.5)
    return False


def start_cloudflared_tunnel(port: int) -> str:
    if not CLOUDFLARED_PATH.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CLOUDFLARED_PATH,
        )
        CLOUDFLARED_PATH.chmod(0o755)

    if "cloudflared" in globals() and cloudflared.poll() is None:
        cloudflared.terminate()
        cloudflared.wait(timeout=5)

    cloudflared_log = CLOUDFLARED_LOG_PATH.open("w")
    process = subprocess.Popen(
        [
            str(CLOUDFLARED_PATH),
            "tunnel",
            "--url",
            f"http://127.0.0.1:{port}",
            "--no-autoupdate",
        ],
        stdout=cloudflared_log,
        stderr=subprocess.STDOUT,
    )

    deadline = time.time() + 30.0
    url_pattern = re.compile(r"https://[-a-zA-Z0-9.]+trycloudflare.com")
    while time.time() < deadline:
        if process.poll() is not None:
            cloudflared_log.flush()
            cloudflared_log.close()
            log_text = CLOUDFLARED_LOG_PATH.read_text() if CLOUDFLARED_LOG_PATH.exists() else "No cloudflared log found."
            raise RuntimeError(f"Cloudflare tunnel exited early.\n\n{log_text}")

        cloudflared_log.flush()
        if CLOUDFLARED_LOG_PATH.exists():
            log_text = CLOUDFLARED_LOG_PATH.read_text()
            match = url_pattern.search(log_text)
            if match:
                globals()["cloudflared"] = process
                globals()["cloudflared_log"] = cloudflared_log
                return match.group(0)
        time.sleep(0.5)

    cloudflared_log.flush()
    cloudflared_log.close()
    log_text = CLOUDFLARED_LOG_PATH.read_text() if CLOUDFLARED_LOG_PATH.exists() else "No cloudflared log found."
    raise RuntimeError(f"Cloudflare tunnel did not start in time.\n\n{log_text}")


if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=5)

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
log_handle = LOG_PATH.open("w")
server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "jupyter",
        "server",
        "--allow-root",
        f"--ServerApp.port={PORT}",
        "--ServerApp.ip=0.0.0.0",
        "--ServerApp.open_browser=False",
        f"--ServerApp.token={TOKEN}",
        "--ServerApp.allow_origin=*",
        "--ServerApp.disable_check_xsrf=True",
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
)

if not wait_for_port("127.0.0.1", PORT):
    log_handle.flush()
    log_handle.close()
    log_text = LOG_PATH.read_text() if LOG_PATH.exists() else "No log file found."
    raise RuntimeError(
        "Jupyter server did not start.\n\n"
        f"Process return code: {server.poll()}\n\n"
        "Startup log:\n"
        f"{log_text}"
    )

log_handle.flush()
try:
    public_url = ngrok.connect(PORT, bind_tls=True).public_url
    tunnel_provider = "ngrok"
except PyngrokNgrokError:
    public_url = start_cloudflared_tunnel(PORT)
    tunnel_provider = "cloudflared"

print("VS Code Jupyter URL:")
print(f"{public_url}?token={TOKEN}")
print()
print(f"Tunnel provider: {tunnel_provider}")
print(f"Runtime directory: {RUNTIME_DIR}")
print(f"Startup log: {LOG_PATH}")
if tunnel_provider == "cloudflared":
    print(f"Tunnel log: {CLOUDFLARED_LOG_PATH}")
print("Keep this Colab session running while VS Code is attached.")

ERROR:pyngrok.process.ngrok:t=2026-04-07T12:10:40+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-07T12:10:40+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-07T12:10:40+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

VS Code Jupyter URL:
https://palace-garden-tuner-equal.trycloudflare.com?token=gpu

Tunnel provider: cloudflared
Runtime directory: /tmp/vscode-colab
Startup log: /tmp/vscode-colab/jupyter-vscode.log
Tunnel log: /tmp/vscode-colab/cloudflared.log
Keep this Colab session running while VS Code is attached.


## Project bootstrap

This clones the project into the Colab runtime and installs dependencies there so your VS Code notebook cells can import from `src/` and access project files after you attach to the remote kernel.

In [4]:
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

try:
    import google.colab  # type: ignore
except ImportError as exc:
    raise RuntimeError(
        "This bootstrap cell must be run in the Colab runtime after Cell 5 has succeeded there. "
        "If you run it locally, it will clone and install into the wrong environment."
    ) from exc

REPO_OWNER = "ncapac"
REPO_NAME = "tesina"
REPO_BRANCH = "master"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
ARCHIVE_URL = f"https://codeload.github.com/{REPO_OWNER}/{REPO_NAME}/zip/refs/heads/{REPO_BRANCH}"
RUNTIME_DIR = Path(tempfile.gettempdir()) / "vscode-colab"
REPO_DIR = RUNTIME_DIR / REPO_NAME
ARCHIVE_PATH = RUNTIME_DIR / f"{REPO_NAME}.zip"
EXTRACT_DIR = RUNTIME_DIR / f"{REPO_NAME}-{REPO_BRANCH}"

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)


def clone_repo() -> None:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "git clone failed")


def download_repo_archive() -> None:
    if ARCHIVE_PATH.exists():
        ARCHIVE_PATH.unlink()
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

    urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
        archive.extractall(RUNTIME_DIR)

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.move(str(EXTRACT_DIR), str(REPO_DIR))


# Clean up a partial clone from a previous failed attempt.
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    try:
        clone_repo()
        print("Repository fetched with git clone.")
    except Exception as clone_exc:
        print(f"git clone failed, falling back to archive download: {clone_exc}")
        download_repo_archive()
        print("Repository fetched from GitHub archive.")
else:
    if (REPO_DIR / ".git").exists():
        result = subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            shutil.rmtree(REPO_DIR)
            try:
                clone_repo()
                print("Repository recloned with git after failed pull.")
            except Exception as clone_exc:
                print(f"git refresh failed, falling back to archive download: {clone_exc}")
                download_repo_archive()
                print("Repository refreshed from GitHub archive.")
        else:
            print("Repository updated with git pull.")
    else:
        shutil.rmtree(REPO_DIR)
        download_repo_archive()
        print("Repository refreshed from GitHub archive.")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Repository fetched with git clone.
Working directory: /tmp/vscode-colab/tesina


In [5]:
import jax
print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX devices: [CudaDevice(id=0)]
Default backend: gpu
